# A1 — Redeveloped LLM (Layer 3)
**COMP8420 2026 S1 — BEACON Brand Monitoring**

Multi-task Q-Former-style RoBERTa: crisis severity, sentiment, and topic from a single Reddit post.

## 1. Objective

**Technique:** A redeveloped multi-task foundation model using a **Q-Former-style architecture** — learnable per-task query tokens plus cross-attention over a **frozen RoBERTa-base encoder** — to jointly predict:

- **Crisis severity** (0–3): 0 = no concern, 1 = minor complaint, 2 = escalating concern, 3 = active crisis
- **Fine-grained sentiment** (−1.0 to 1.0, regression head)
- **Topic** (fixed 8-category brand-monitoring taxonomy)

**Why this matters for social media brand monitoring:** One shared encoder pass per post yields three specialised predictions, avoiding three separate models at inference time. Per-task query tokens let each head extract task-relevant features from the full token sequence instead of competing for a single `[CLS]` vector — reducing task interference in multi-task learning.

## 2. Setup

Imports, reproducibility seeds, data loading from B1's clean corpus, **committed pseudo-labels**, eval-set files, and train/val split.

This notebook is **deterministic**: it does not call any LLM at runtime. Training labels come from `pseudo_labels.csv`, produced once via external **ChatGPT** labeling (see workflow below).

**Manual review gate:** `eval_set_reviewed.csv` must exist before Sections 4–6.

In [ ]:
# Install dependencies (uncomment on first run)
# !pip install torch transformers datasets scikit-learn pandas numpy matplotlib tqdm -q

In [ ]:
import json
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer

ROOT = Path.cwd().resolve()
if ROOT.name == "advanced":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from shared.data_loader import load_clean_corpus
from advanced.layer3_llm.baseline_compare import (
    evaluate_pipeline_baseline,
    measure_pipeline_latency_ms,
    pipeline_trainable_param_count,
)
from advanced.layer3_llm.labeling import export_labeling_batch, load_pseudo_labels
from advanced.layer3_llm.multitask_models import (
    LOSS_WEIGHTS,
    QFormerMultiTaskRoberta,
    TOPIC_LABELS,
)
from advanced.layer3_llm.train_utils import (
    MultiTaskDataset,
    evaluate_predictions,
    get_device,
    load_checkpoint,
    measure_inference_latency_ms,
    predict_batch,
    save_checkpoint,
    train_model,
)

LAYER3_DIR = ROOT / "advanced" / "layer3_llm"
OUTPUT_DIR = LAYER3_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROMPT_PATH = LAYER3_DIR / "chatgpt_labeling_prompt.md"
POSTS_JSON = OUTPUT_DIR / "posts_for_labeling.json"
PSEUDO_CSV = OUTPUT_DIR / "pseudo_labels.csv"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = get_device()
print(f"Project root: {ROOT}")
print(f"Output dir:   {OUTPUT_DIR}")
print(f"Device:       {DEVICE}  (set LAYER3_DEVICE=cpu|cuda to override)")
print(f"ChatGPT prompt: {PROMPT_PATH}")

In [ ]:
df = load_clean_corpus(brand="openai", as_df=True)
TEXT_COL = "text_for_llm"
df = df[df[TEXT_COL].astype(str).str.len() > 20].reset_index(drop=True)
df[TEXT_COL] = df[TEXT_COL].astype(str).str.slice(0, 1500)

display(df[["post_id", TEXT_COL]].head(3))
print(f"\nCorpus: {len(df)} posts")

### External ChatGPT labeling (one-time, outside this notebook)

We do **not** pseudo-label inside the notebook. Instead, labels are produced once in ChatGPT and committed to the repo so every notebook run is identical.

**Workflow:**

1. **Export batch** — run the cell below once. It writes `outputs/posts_for_labeling.json` (500 posts, `SEED=42`). This file is committed to the repo.
2. **Open** [`advanced/layer3_llm/chatgpt_labeling_prompt.md`](layer3_llm/chatgpt_labeling_prompt.md) and copy the full prompt text into ChatGPT.
3. **Append** the contents of `posts_for_labeling.json` immediately after the prompt.
4. **ChatGPT returns** a JSON array of label objects. Convert to CSV with columns:
   `post_id, text, crisis_severity, sentiment_score, topic, rationale`
   (copy `text` from the input JSON; one row per post, same order).
5. **Save** as `advanced/layer3_llm/outputs/pseudo_labels.csv` and commit to the repo.
   *(Or save ChatGPT's JSON array to a file and run `python scripts/merge_chatgpt_labels.py chatgpt_response.json`.)*
6. **Re-run** this notebook — it loads the CSV deterministically; no API calls.

If `pseudo_labels.csv` already exists in the repo, skip steps 1–5.

In [ ]:
# Step 1 — export labeling batch (run once; skip if posts_for_labeling.json is already committed)
if not POSTS_JSON.exists():
    export_labeling_batch(df, POSTS_JSON, text_col=TEXT_COL, n=500, seed=SEED)
    print(f"Created {POSTS_JSON}")
else:
    batch = json.loads(POSTS_JSON.read_text())
    print(f"Using committed batch: {POSTS_JSON} ({len(batch)} posts)")
    print(f"Open {PROMPT_PATH.name} → paste into ChatGPT with this JSON")

In [ ]:
# Step 6 — load committed pseudo-labels (required; notebook fails clearly if missing)
pseudo_df = load_pseudo_labels(PSEUDO_CSV)
print(f"Loaded {PSEUDO_CSV} ({len(pseudo_df)} rows)")
print(f"Label source: external ChatGPT (see {PROMPT_PATH.name})")
print("\nSeverity distribution:")
display(pseudo_df["crisis_severity"].value_counts().sort_index())
print("\nTopic distribution:")
display(pseudo_df["topic"].value_counts())
display(pseudo_df[["post_id", "crisis_severity", "sentiment_score", "topic"]].head(5))

In [ ]:
EVAL_TO_REVIEW = OUTPUT_DIR / "eval_set_to_review.csv"
EVAL_REVIEWED = OUTPUT_DIR / "eval_set_reviewed.csv"

# Deterministic eval holdout — load committed file or create once then commit
if EVAL_TO_REVIEW.exists():
    eval_df = pd.read_csv(EVAL_TO_REVIEW)
    print(f"Loaded committed eval set: {EVAL_TO_REVIEW} ({len(eval_df)} posts)")
else:
    N_EVAL = min(250, len(pseudo_df))
    weights = pseudo_df["crisis_severity"].map({0: 1.0, 1: 1.0, 2: 2.0, 3: 2.0}).fillna(1.0)
    eval_df = pseudo_df.sample(n=N_EVAL, weights=weights, random_state=SEED).copy()
    eval_df["human_verified"] = ""
    eval_df.to_csv(EVAL_TO_REVIEW, index=False)
    print(f"Created {EVAL_TO_REVIEW} — review labels, save as eval_set_reviewed.csv, commit both")

print("\nEval severity distribution:")
display(eval_df["crisis_severity"].value_counts().sort_index())

eval_ids = set(eval_df["post_id"].astype(str))
train_pool = pseudo_df[~pseudo_df["post_id"].astype(str).isin(eval_ids)].copy()
train_df, val_df = train_test_split(
    train_pool,
    test_size=0.15,
    random_state=SEED,
    stratify=train_pool["crisis_severity"],
)
assert eval_ids.isdisjoint(set(train_df["post_id"].astype(str)))
assert eval_ids.isdisjoint(set(val_df["post_id"].astype(str)))
print(f"\nTrain: {len(train_df)} | Val: {len(val_df)} | Eval holdout: {len(eval_ids)}")

### ⏸ Manual review required

1. Open `advanced/layer3_llm/outputs/eval_set_to_review.csv`
2. Correct `crisis_severity`, `sentiment_score`, and `topic` where ChatGPT pseudo-labels were wrong
3. Set `human_verified` to `yes` for reviewed rows
4. Save as `advanced/layer3_llm/outputs/eval_set_reviewed.csv` and commit

**Do not proceed to Section 4 until the reviewed file exists.**

In [ ]:
REVIEWED = OUTPUT_DIR / "eval_set_reviewed.csv"
assert REVIEWED.exists(), (
    "Manual review required: correct eval_set_to_review.csv and save as eval_set_reviewed.csv"
)
eval_gold = pd.read_csv(REVIEWED)
eval_ids = set(eval_gold["post_id"])
print(f"Loaded reviewed eval set: {len(eval_gold)} posts")

## 3. Implementation

**Q-Former-style multi-task RoBERTa:** frozen `roberta-base` encoder; three task towers each with 6 learnable query tokens and 2 cross-attention layers; mean-pooled query outputs feed task heads.

- `crisis_head`: 4-way softmax (CrossEntropy)
- `sentiment_head`: scalar regression (MSE, clamped to [−1, 1] at inference)
- `topic_head`: 8-way softmax (CrossEntropy)

Combined loss = weighted sum (crisis 1.0, sentiment 0.5, topic 0.5). Only query towers, cross-attention, and heads are trainable.

*Note:* We keep the encoder fully frozen to isolate the query-tower benefit and stay under the 1-hour runtime budget. Partial unfreezing of the last 1–2 RoBERTa layers could help Reddit slang but adds train time and is left as future work.

In [ ]:
MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-4
MODEL_NAME = "roberta-base"

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
train_ds = MultiTaskDataset(train_df, tokenizer, text_col="text", max_length=MAX_LENGTH)
val_ds = MultiTaskDataset(val_df, tokenizer, text_col="text", max_length=MAX_LENGTH)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

qformer = QFormerMultiTaskRoberta(model_name=MODEL_NAME)
print(f"Q-Former trainable parameters: {qformer.count_trainable_params():,}")

qformer_history, qformer_train_time = train_model(
    qformer, train_loader, val_loader, epochs=EPOCHS, lr=LR, device=DEVICE
)

qformer_config = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "num_queries": 6,
    "num_xattn_layers": 2,
    "loss_weights": LOSS_WEIGHTS,
}
save_checkpoint(
    OUTPUT_DIR / "qformer_model.pt",
    qformer,
    arch="qformer",
    config=qformer_config,
    history=qformer_history,
)
print(f"Checkpoint saved. Train time: {qformer_train_time:.1f}s")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
metrics = [
    ("train_loss", "val_loss", "Total loss"),
    ("train_loss_crisis", "val_loss_crisis", "Crisis loss"),
    ("train_loss_sentiment", "val_loss_sentiment", "Sentiment loss"),
    ("train_loss_topic", "val_loss_topic", "Topic loss"),
]
for ax, (tk, vk, title) in zip(axes.flat[:4], metrics):
    ax.plot(qformer_history[tk], label="train")
    ax.plot(qformer_history[vk], label="val")
    ax.set_title(title)
    ax.legend()
    ax.set_xlabel("epoch")
for ax in axes.flat[4:]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "qformer_loss_curves.png", dpi=120)
plt.show()

## 4. Results

Evaluate the trained Q-Former on the human-reviewed holdout set.

In [ ]:
eval_ds = MultiTaskDataset(eval_gold, tokenizer, text_col="text", max_length=MAX_LENGTH)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False)

ckpt = load_checkpoint(OUTPUT_DIR / "qformer_model.pt", DEVICE)
qformer.load_state_dict(ckpt["state_dict"])
qformer.to(DEVICE)

preds = predict_batch(qformer, eval_loader, DEVICE)
topic_to_id = {t: i for i, t in enumerate(TOPIC_LABELS)}
y_topic_true = [topic_to_id.get(str(t), topic_to_id["general_discussion"]) for t in eval_gold["topic"]]

qformer_metrics = evaluate_predictions(
    eval_gold["crisis_severity"].tolist(), preds["crisis_pred"],
    eval_gold["sentiment_score"].tolist(), preds["sentiment_pred"],
    y_topic_true, preds["topic_pred"],
)
metrics_df = pd.DataFrame([qformer_metrics]).drop(columns=["crisis_confusion_matrix"])
display(metrics_df.T)
metrics_df.to_csv(OUTPUT_DIR / "qformer_eval_metrics.csv", index=False)

cm = np.array(qformer_metrics["crisis_confusion_matrix"])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[0,1,2,3], yticklabels=[0,1,2,3])
plt.xlabel("Predicted severity"); plt.ylabel("True severity"); plt.title("Q-Former crisis confusion matrix")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "qformer_confusion_matrix.png", dpi=120)
plt.show()

In [ ]:
examples = eval_gold.copy()
examples["pred_crisis"] = preds["crisis_pred"]
examples["pred_sentiment"] = np.round(preds["sentiment_pred"], 3)
examples["pred_topic"] = [TOPIC_LABELS[i] for i in preds["topic_pred"]]
examples["crisis_ok"] = examples["crisis_severity"] == examples["pred_crisis"]
examples["topic_ok"] = examples["topic"] == examples["pred_topic"]
examples["sent_err"] = (examples["sentiment_score"] - examples["pred_sentiment"]).abs()

successes = examples[examples["crisis_ok"] & examples["topic_ok"] & (examples["sent_err"] < 0.3)].head(3)
failures = examples.sort_values("sent_err", ascending=False).head(3)
show = pd.concat([successes, failures]).drop_duplicates("post_id").head(10)
show["text_snip"] = show["text"].str.slice(0, 120) + "..."
display(show[["text_snip", "crisis_severity", "pred_crisis", "sentiment_score", "pred_sentiment", "topic", "pred_topic"]])

## 5. Comparison

Compare the **Q-Former multi-task model** (one forward pass, three heads) against the **existing BEACON pipeline** assembled from basic-layer components:

| Task | Pipeline baseline |
|------|-------------------|
| Crisis severity | **TF-IDF + Logistic Regression** (trained on ChatGPT pseudo-labels) |
| Sentiment | **B4** — `classify_sentiment_detailed()` → `basic/sentiment_model.pkl` + `tfidf_vectorizer.pkl` |
| Topic | **B5** — `fit_bertopic()` + `assign_thread_topics()` from `shared/topics.py` |

No dashboard JSON stubs — we run the same exported functions and model artefacts as B4/B5.

In [ ]:
CRISIS_LR_CKPT = OUTPUT_DIR / "crisis_lr_baseline.pkl"
BERTOPIC_CKPT = OUTPUT_DIR / "bertopic_b5_baseline"

pipeline_metrics, crisis_vec, crisis_clf, bertopic_model, pipeline_train_time = (
    evaluate_pipeline_baseline(
        eval_gold,
        train_pool,
        crisis_ckpt=CRISIS_LR_CKPT,
        bertopic_ckpt=BERTOPIC_CKPT,
        random_state=SEED,
    )
)
print(f"Pipeline fit time (crisis LR + B5 BERTopic): {pipeline_train_time:.2f}s")
print(f"Crisis LR coefficients: {pipeline_trainable_param_count(crisis_vec, crisis_clf):,}")
print(f"B5 topics discovered: {len(bertopic_model.get_topic_info()) - 1}")  # minus outlier row

qformer_latency = measure_inference_latency_ms(qformer, eval_loader, DEVICE)
pipeline_latency = measure_pipeline_latency_ms(
    eval_gold, crisis_vec, crisis_clf, bertopic_model, text_col="text"
)

comparison = pd.DataFrame([
    {
        "model": "Q-Former multi-task",
        "crisis_f1": qformer_metrics["crisis_f1_macro"],
        "sentiment_mae": qformer_metrics["sentiment_mae"],
        "topic_f1": qformer_metrics["topic_f1_macro"],
        "avg_score": qformer_metrics["avg_score"],
        "trainable_params": qformer.count_trainable_params(),
        "train_time_s": round(qformer_train_time, 1),
        "inference_ms_per_post": round(qformer_latency, 2),
    },
    {
        "model": "B4 + B5 + crisis LR (pipeline)",
        "crisis_f1": pipeline_metrics["crisis_f1_macro"],
        "sentiment_mae": pipeline_metrics["sentiment_mae"],
        "topic_f1": pipeline_metrics["topic_f1_macro"],
        "avg_score": pipeline_metrics["avg_score"],
        "trainable_params": pipeline_trainable_param_count(crisis_vec, crisis_clf),
        "train_time_s": round(pipeline_train_time, 1),
        "inference_ms_per_post": round(pipeline_latency, 2),
    },
])
display(comparison)
comparison.to_csv(OUTPUT_DIR / "comparison_table.csv", index=False)

print("\nPer-component note: B4 uses basic/sentiment_model.pkl; B5 fits BERTopic on train pool; crisis LR fit here.")

In [ ]:
labels = ["Crisis F1", "Topic F1", "Sentiment (1-MAE/2)"]
x = np.arange(len(labels))
width = 0.35
q_scores = [
    qformer_metrics["crisis_f1_macro"],
    qformer_metrics["topic_f1_macro"],
    1 - qformer_metrics["sentiment_mae"] / 2,
]
p_scores = [
    pipeline_metrics["crisis_f1_macro"],
    pipeline_metrics["topic_f1_macro"],
    1 - pipeline_metrics["sentiment_mae"] / 2,
]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(x - width/2, q_scores, width, label="Q-Former")
axes[0].bar(x + width/2, p_scores, width, label="B4+B5+LR pipeline")
axes[0].set_xticks(x); axes[0].set_xticklabels(labels, rotation=15)
axes[0].set_ylim(0, 1); axes[0].set_title("Per-task scores"); axes[0].legend()

eff_labels = ["Train time (s)", "Latency (ms/post)"]
q_eff = [qformer_train_time, qformer_latency]
p_eff = [pipeline_train_time, pipeline_latency]
xe = np.arange(len(eff_labels))
axes[1].bar(xe - width/2, q_eff, width, label="Q-Former")
axes[1].bar(xe + width/2, p_eff, width, label="Pipeline")
axes[1].set_xticks(xe); axes[1].set_xticklabels(eff_labels)
axes[1].set_title("Efficiency comparison"); axes[1].legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "comparison_charts.png", dpi=120)
plt.show()

## 6. Justification

Evidence-based comparison for the report (auto-generated from Section 5 results).

In [ ]:
q_row = comparison[comparison["model"] == "Q-Former multi-task"].iloc[0]
p_row = comparison[comparison["model"] == "B4 + B5 + crisis LR (pipeline)"].iloc[0]

crisis_winner = "Q-Former" if q_row["crisis_f1"] >= p_row["crisis_f1"] else "Pipeline (crisis LR)"
topic_winner = "Q-Former" if q_row["topic_f1"] >= p_row["topic_f1"] else "B5 topics"
latency_winner = "Q-Former" if q_row["inference_ms_per_post"] <= p_row["inference_ms_per_post"] else "Pipeline"

justification = f"""
**Paragraph 1 — Task performance:** On the human-reviewed eval set ({len(eval_gold)} posts), Q-Former achieves crisis macro-F1={q_row['crisis_f1']:.3f} vs {p_row['crisis_f1']:.3f} for TF-IDF+LR crisis detection; topic macro-F1={q_row['topic_f1']:.3f} vs {p_row['topic_f1']:.3f} for B5 BERTopic (shared.topics); sentiment MAE={q_row['sentiment_mae']:.3f} vs {p_row['sentiment_mae']:.3f} for B4 LR sentiment. Overall average score: {q_row['avg_score']:.3f} vs {p_row['avg_score']:.3f}. {crisis_winner} is stronger on crisis — the task with no existing basic-layer module.

**Paragraph 2 — Unified vs modular pipeline:** Q-Former runs one encoder pass for all three tasks. The pipeline baseline stitches together three separate basic-layer components (B4 LR pkl, B5 BERTopic via fit_bertopic/assign_thread_topics, crisis LR trained in ~{p_row['train_time_s']:.0f}s on pseudo-labels). Q-Former training took {q_row['train_time_s']:.0f}s but replaces three inference paths with one. Inference: {q_row['inference_ms_per_post']:.1f} ms/post (Q-Former) vs {p_row['inference_ms_per_post']:.1f} ms/post (pipeline); {latency_winner} is faster per post.

**Paragraph 3 — Brand monitoring relevance:** Crisis severity has no B-layer equivalent until this notebook — the LR baseline validates that a basic technique on pseudo-labels underperforms (or matches) the Q-Former, justifying a unified model for real-time scoring via predict(). Severity ≥2 triggers ReAct/CoT and RAG in Phase 2.
"""
print(justification)

## 7. Limitations

The frozen RoBERTa encoder was not pretrained on contemporary Reddit/OpenAI discourse, so sarcasm, memes, and community-specific slang may be misread. Training labels come from **ChatGPT pseudo-labeling** (external, one-time) and inherit whatever bias or blind spots that model has. Severity-3 crises are rare even after stratified oversampling, so real-world recall on true crises remains uncertain beyond what the 200–500 post human-reviewed eval set can show.

## 8. Pipeline connection

Layer 2's B1 output (`text_for_llm` from `reddit_openai_clean.jsonl`) feeds this notebook as input text. The exported `predict()` function in `advanced/layer3_llm/predict.py` is called by Layer 4's crisis monitor (A5 ReAct) to score incoming posts in real time. Posts with `crisis_severity` above a configured threshold trigger ReAct/CoT/ToT reasoning and A2 RAG retrieval for evidence-backed crisis reports in Phase 2.

## 9. Export function

Phase 2 imports `predict()` directly — no re-training required.

In [ ]:
from advanced.layer3_llm.predict import predict

samples = [
    "OpenAI API has been down for 6 hours, our production app is completely broken.",
    "GPT-4o voice mode is incredible, best update in months!",
    "Subscription price keeps going up, not sure it's worth it anymore.",
]
for s in samples:
    print(json.dumps({"text": s[:80], **predict(s)}, indent=2))
    print()